# Produção · previsão da TEMPORADA (SE41 → SE40, com dados até a SE25)

Roda os **5 melhores modelos de temporada** (WIS e MAE no teste 2022–2025) para a temporada mais recente cuja SE25 já está nos dados:

| # | Modelo | WIS | MAE | Altura do pico | Total |
|---|---|---|---|---|---|
| 1 | **ENS_pico7_simples_UF** (campeão) | 721 | 1.277 | +2% | −0,6% |
| 2 | MIX_escala5_UF | 746 | 1.279 | +12% | +9% |
| 3 | ENS_pico7_UF | 902 | 1.452 | −20% | −16% |
| 4 | MIX_elastico5_UF | 1.010 | 1.506 | −12% | −8% |
| 5 | AJ_continuo_UF | 1.285 | 1.672 | −28% | −25% |

**Cadeia:**
1. **Normal** (`ENS_grafos3_λ`): mediana de 4 redes de grafo (MTGNN, A3T-GCN e GCNGRU com λ = 0,8; MTGNN do Optuna com 10 sementes). Precisa de **GPU**.
2. **Detector** de anos grandes: ridge do log do tamanho relativo ao normal da UF, com sinais escolhidos só até 2021.
3. **Misturas:** elástica (w = 0,5), de escala (limiar 3×) e contínua (p = 0,25).
4. **Ensembles do pico** (rodada 7).

`NORMAL_SEM_GPU = True` troca o normal pela climatologia **só para testar o fluxo**: não é o modelo campeão.

**Saídas** em `producao_temporada/`: `previsao_temporada_<s>.parquet` (top 5 + normal), `detector_<s>.csv` e `resumo_temporada_<s>.csv` (campeão, por UF).


In [ ]:
PACOTES = ''

# @title 1 · Área de trabalho (Drive no Colab ou pasta local), pacotes e contexto
import os, sys, shutil, glob, json, time, subprocess, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/prod', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False          # Jupyter aberto na área de trabalho
if NO_COLAB:
    if PACOTES:
        subprocess.run(f'pip install -q {PACOTES}', shell=True, check=True)
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py', 'r4_lib.py',
                 'r5_lib.py', 'r6_lib.py', 'r7_lib.py', 'prod_lib.py', 'relatorio_campeoes.py', 'graficos_previsoes.py',
                 'dados', 'dados_uf', 'referencias_imdc3', 'parametros'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou {item} em {PASTA_DRIVE}: rode scripts/04_montar_area_de_trabalho.py e suba a pasta'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
sys.path.insert(0, PASTA_LOCAL)
import importlib, v4_lib as V, comp_lib as L, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3, prod_lib as PL
for m in (V, L, U, S, R2, R3, PL):
    importlib.reload(m)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 120)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'), os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
P = {k: os.path.join(PASTA_DRIVE, v) for k, v in dict(PS='producao_semanal', PT='producao_temporada', REL='relatorio_campeoes',
                                                      CACHE='cache_producao').items()}
for p in P.values():
    os.makedirs(p, exist_ok=True)
PAR = PL.ler_parametros(os.path.join(PASTA_LOCAL, 'parametros'))
ULTIMA = pd.Timestamp(D.datas[D.ultimo_obs]).date()
print('última semana observada:', ULTIMA, '· UFs:', D.N)


In [ ]:
# @title 2 · Temporada-alvo e normal (4 redes de grafo, GPU; retoma)
S_ALVO = PL.temporada_alvo(D)
RODAR_BACKTEST = False                      # True: também as temporadas do teste (2022–2025), para conferir as métricas
NORMAL_SEM_GPU = False                      # True: climatologia no lugar do normal (só para testar o fluxo)
TEMPORADAS = sorted(set([S_ALVO] + (TESTE if RODAR_BACKTEST else [])))
print('temporada-alvo:', S_ALVO, f'({S_ALVO}/{S_ALVO + 1}, prevista com dados até a SE25 de {S_ALVO}) · temporadas:', TEMPORADAS)
if NORMAL_SEM_GPU:
    print('⚠️ normal = climatologia (teste do fluxo; NÃO é o modelo campeão)')
    NORMAL = PL.normal_climatologia(D, TEMPORADAS)
else:
    PASTA_REDES = os.path.join(P['PT'], 'redes')
    PL.rodar_normal(ctx, PL.modelos_normal(PAR, os.path.join(P['CACHE'], 'ckpt_temporada')), TEMPORADAS, PASTA_REDES)
    NORMAL = PL.normal(PASTA_REDES, TEMPORADAS)
NORMAL.groupby('season').size()


In [ ]:
# @title 3 · Detector → misturas → ensembles do pico
X, DET = PL.prever_temporada(ctx, NORMAL, TEMPORADAS, P['CACHE'])
for s in TEMPORADAS:
    PL.gravar(X[X.season == s], P['PT'], f'previsao_temporada_{s}.parquet')
    PL.gravar(DET[DET.season == s], P['PT'], f'detector_{s}.csv')
RES = PL.resumo_temporada(X, D, S_ALVO)
d = DET[DET.season == S_ALVO].assign(uf=lambda x: x.uf_code.map(R2.SIGLAS), tamanho_x_normal=lambda x: np.exp(x.pred))
RES = RES.merge(d[['uf', 'p_grande', 'tamanho_x_normal']], on='uf', how='left')
PL.gravar(RES, P['PT'], f'resumo_temporada_{S_ALVO}.csv')
print(f'{PL.CAMPEAO_TEMPORADA} · temporada {S_ALVO}/{S_ALVO + 1}: total, semana e altura do pico por UF; P(ano grande) e tamanho previsto pelo detector')
RES.round(2)


In [ ]:
# @title 4 · (se RODAR_BACKTEST) métricas no teste 2022–2025
if RODAR_BACKTEST:
    OF = pd.concat([X[X.season.isin(TESTE)], ctx.ref[ctx.ref.modelo == 'BB'][L.COLS_PREV]], ignore_index=True)
    _, R = V.avaliar(OF[L.COLS_PREV], ctx.ver_uf, ref=('BB',))
    TT = S.com_verdade(OF, D)
    PT = pd.DataFrame([dict(modelo=m, **S.metricas(g)) for m, g in TT.groupby('modelo')])
    TAB = R[['modelo', 'R_vs_BB', 'UFs_melhor_que_BB']].merge(PT, on='modelo').sort_values('WIS')
    PL.gravar(TAB, P['PT'], 'backtest_metricas.csv')
    display(TAB[['modelo', 'R_vs_BB', 'UFs_melhor_que_BB', 'WIS', 'MAE', 'WAPE', 'sMAPE', 'IC90']].round(3))
